# Zotero group overview

Reads the JSON generated by `zotsync.cli` (or fetches directly from Zotero, in the optional section) and shows
the library overview: size, years, journals, collections, tags and metadata quality.

**Read-only**: nothing here writes to Zotero.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Allows importing zotsync when the notebook runs from inside notebooks/
sys.path.insert(0, str(Path.cwd().parent / "src"))

# --- Style: one color for magnitude, neutral ink for text, subtle grid ---
BAR = "#2A6FBB"
INK, MUTED, GRID = "#1F2933", "#6B7785", "#E4E7EB"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "left",
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
})
pd.set_option("display.max_colwidth", 80)

## 1. Load the data

In [ ]:
DATA = Path("../data/zotero_refs.json")   # generated by: python -m zotsync.cli
payload = json.loads(DATA.read_text(encoding="utf-8"))
df = pd.DataFrame(payload["items"])
print(f"{len(df)} references | library version: {payload['library_version']}")
df.head()

### (Optional) Fetch directly from Zotero

Run this cell instead of the previous one if you prefer not to go through the JSON.
Requires `ZOTERO_API_KEY` and `ZOTERO_GROUP_ID` in the environment.

In [ ]:
# from zotsync.reader import fetch_group
# refs, version = fetch_group(os.environ["ZOTERO_GROUP_ID"], os.environ["ZOTERO_API_KEY"])
# df = pd.DataFrame([r.to_dict() for r in refs])
# print(len(df), "references, version", version)

## 2. Metadata quality

What is missing in each reference. The DOI is the planned join key for Notion.

In [ ]:
checks = pd.Series({
    "with DOI": (df.doi != "").sum(),
    "with year": df.year.notna().sum(),
    "with abstract": (df.abstract != "").sum(),
    "with PDF": df.has_pdf.sum(),
    "in some collection": (df.collections.str.len() > 0).sum(),
    "with tags": (df.tags.str.len() > 0).sum(),
})
pct = (checks / len(df) * 100).round(0).astype(int)

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(checks.index[::-1], pct[::-1], color=BAR, height=0.55)
for y, (v, n) in enumerate(zip(pct[::-1], checks[::-1])):
    ax.text(v + 1.5, y, f"{v}% ({n})", va="center", color=INK, fontsize=9)
ax.set_xlim(0, 115); ax.set_xlabel("% of references"); ax.grid(axis="y", visible=False)
ax.set_title("Metadata completeness")
plt.show()

dup = df[df.doi != ""].doi.value_counts()
dup = dup[dup > 1]
print(f"Duplicated DOIs: {len(dup)}")
if len(dup):
    display(df[df.doi.isin(dup.index)].sort_values("doi")[["key", "doi", "title", "year"]])

## 3. Publications per year

In [ ]:
by_year = df.year.dropna().astype(int).value_counts().sort_index()
by_year = by_year.reindex(range(by_year.index.min(), by_year.index.max() + 1), fill_value=0)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(by_year.index, by_year.values, color=BAR, width=0.7)
ax.set_title("References per publication year"); ax.set_ylabel("no. of references")
ax.grid(axis="x", visible=False)
plt.show()
print("no year:", df.year.isna().sum())

## 4. Journals and item types

In [ ]:
top = df[df.publication != ""].publication.value_counts().head(15)[::-1]
fig, ax = plt.subplots(figsize=(8, 0.35 * len(top) + 1))
ax.barh(top.index, top.values, color=BAR, height=0.6)
for y, v in enumerate(top.values):
    ax.text(v + 0.1, y, str(v), va="center", color=INK, fontsize=9)
ax.set_title("Most frequent journals / sources"); ax.grid(axis="y", visible=False)
plt.show()

df.item_type.value_counts().rename("n").to_frame()

## 5. Collections and tags

An item can be in several collections/tags, so the counts do not add up to the total.

In [ ]:
def count_exploded(col, top_n=20):
    s = df[col].explode().dropna()
    return s.value_counts().head(top_n)[::-1]

for col, title in [("collections", "Items per collection"), ("tags", "Most used tags")]:
    s = count_exploded(col)
    if s.empty:
        print(f"({title}: nothing to show)"); continue
    fig, ax = plt.subplots(figsize=(8, 0.33 * len(s) + 1))
    ax.barh(s.index, s.values, color=BAR, height=0.6)
    for y, v in enumerate(s.values):
        ax.text(v + 0.1, y, str(v), va="center", color=INK, fontsize=9)
    ax.set_title(title); ax.grid(axis="y", visible=False)
    plt.show()

## 6. Browsable table

Filter by text in the title, year or collection and export the subset.

In [ ]:
def view(text="", year_min=None, collection=""):
    m = pd.Series(True, index=df.index)
    if text:        m &= df.title.str.contains(text, case=False, na=False) | df.abstract.str.contains(text, case=False, na=False)
    if year_min:    m &= df.year >= year_min
    if collection:  m &= df.collections.apply(lambda cs: any(collection.lower() in c.lower() for c in cs))
    out = df[m].copy()
    out["authors"] = out.authors.apply(lambda a: "; ".join(a[:3]) + (" et al." if len(a) > 3 else ""))
    return out[["year", "authors", "title", "publication", "doi", "has_pdf"]].sort_values("year", ascending=False)

view(text="", year_min=None, collection="")  # e.g.: view(text="shrinkage", year_min=2015)

## Next step

This same table (`df`) is what step 2 will send to Notion, using the DOI as the join key.